In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

home_data = pd.read_csv("data/train.csv")

y = home_data.SalePrice
features = [
    'LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 
    'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd'
]
X = home_data[features]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

dt_model = DecisionTreeRegressor(random_state=1)
dt_model.fit(train_X, train_y)
val_predictions_dt = dt_model.predict(val_X)
dt_mae = mean_absolute_error(val_y, val_predictions_dt)

def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=1)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    return mean_absolute_error(val_y, preds_val)

candidate_max_leaf_nodes = [5, 25, 50, 100, 250, 500]
scores = {leaf_size: get_mae(leaf_size, train_X, val_X, train_y, val_y) for leaf_size in candidate_max_leaf_nodes}
best_tree_size = min(scores, key=scores.get)

final_dt_model = DecisionTreeRegressor(max_leaf_nodes=best_tree_size, random_state=1)
final_dt_model.fit(X, y)

rf_val_model = RandomForestRegressor(random_state=1)
rf_val_model.fit(train_X, train_y)
rf_val_predictions = rf_val_model.predict(val_X)
rf_val_mae = mean_absolute_error(val_y, rf_val_predictions)

rf_model_on_full_data = RandomForestRegressor(random_state=1)
rf_model_on_full_data.fit(X, y)

test_data = pd.read_csv("data/test.csv")

test_X = test_data[features]

dt_test_preds = final_dt_model.predict(test_X)
rf_test_preds = rf_model_on_full_data.predict(test_X)

print("--- VALIDATION MAE RESULTS (Lower is Better) ---")
print(f"Default Decision Tree MAE:        ${dt_mae:,.0f}")
print(f"Tuned Decision Tree ({best_tree_size} leaves): ${scores[best_tree_size]:,.0f}")
print(f"Random Forest MAE:                ${rf_val_mae:,.0f}\n")

comparison_df = pd.DataFrame({
    'Id': test_data.Id,
    'DecisionTree_Pred': dt_test_preds.round(2),
    'RandomForest_Pred': rf_test_preds.round(2)
})
print("--- SAMPLE TEST PREDICTIONS (First 10 Houses) ---")
print(comparison_df.head(10).to_string(index=False))

output = pd.DataFrame({'Id': test_data.Id, 'SalePrice': rf_test_preds})
output.to_csv('submission.csv', index=False)


--- VALIDATION MAE RESULTS (Lower is Better) ---
Default Decision Tree MAE:        $29,653
Tuned Decision Tree (100 leaves): $27,283
Random Forest MAE:                $21,857

--- SAMPLE TEST PREDICTIONS (First 10 Houses) ---
  Id  DecisionTree_Pred  RandomForest_Pred
1461          125362.57          122656.58
1462          157249.18          156789.00
1463          182392.32          182959.00
1464          182392.32          178102.00
1465          190290.71          189049.48
1466          173237.50          180979.45
1467          190290.71          172797.00
1468          182392.32          173716.72
1469          190290.71          187535.19
1470          125362.57          116171.50
